# Session 2: every collision between sticking and bouncing
**A. Real-world problem.** A basketball dropped from 1 m comes back to 0.64 m. A bullet
buried in a hanging block makes it swing up a few centimetres. What happens in collisions
that are neither perfectly sticky nor perfectly bouncy, how much energy disappears, and
how can a swing tell us the speed of a bullet?

Educational model: straight line, no friction, instant collision. Not a tool for real crash analysis.

In [ ]:
import pathlib

import numpy as np

OUT = (pathlib.Path(__file__).resolve().parent if "__file__" in globals() else pathlib.Path(".")) / "out"
G = 9.81   # gravitational acceleration (m/s^2)

## Step 1: the coefficient of restitution
The speed of approach before the collision and the speed of separation after it are linked by
one number e between 0 and 1:

    e = (v2' - v1') / (v1 - v2)

e = 1 is a perfectly bouncing collision (energy kept), e = 0 is sticking. Together with
momentum conservation this gives both final speeds:

    v1' = (m1 v1 + m2 v2 + m2 e (v2 - v1)) / (m1 + m2)
    v2' = (m1 v1 + m2 v2 + m1 e (v1 - v2)) / (m1 + m2)

In [ ]:
def collide(m1, v1, m2, v2, e):
    """Speeds (v1', v2') after a 1D collision with restitution e (0 to 1)."""
    if m1 <= 0 or m2 <= 0:
        raise ValueError("masses must be positive")
    if not 0 <= e <= 1:
        raise ValueError("e must be between 0 and 1")
    s = m1 * v1 + m2 * v2
    return (s + m2 * e * (v2 - v1)) / (m1 + m2), (s + m1 * e * (v1 - v2)) / (m1 + m2)

## Step 2: how much energy is lost
Write the reduced mass mu = m1 m2 / (m1 + m2). The kinetic energy that disappears (into
heat, sound and deformation) depends only on mu, on e and on the relative speed:

    dKE = 1/2 mu (1 - e^2) (v1 - v2)^2

In [ ]:
def energy_lost(m1, v1, m2, v2, e):
    mu = m1 * m2 / (m1 + m2)
    return 0.5 * mu * (1 - e ** 2) * (v1 - v2) ** 2


def ke(m, v):
    return 0.5 * m * v ** 2

## Step 3: two measurements
**Bouncing ball.** A ball dropped from height h and rebounding to h' hits the floor with
speed sqrt(2 g h) and leaves with sqrt(2 g h'). The floor does not move, so

    e = sqrt(h' / h)

**Ballistic pendulum.** A bullet of mass m is buried in a block of mass M hanging on strings.
The sticking collision (momentum) is followed by a swing up to a height h (energy):

    v_bullet = (m + M) / m x sqrt(2 g h)

In [ ]:
def restitution_from_heights(h_drop, h_rebound):
    if h_drop <= 0 or h_rebound < 0 or h_rebound > h_drop:
        raise ValueError("need 0 <= rebound height <= drop height, and a positive drop height")
    return (h_rebound / h_drop) ** 0.5


def bullet_speed(m, big_m, h):
    """Speed of a bullet of mass m (kg) embedded in a block of mass big_m (kg) that rises h metres."""
    return (m + big_m) / m * (2 * G * h) ** 0.5

In [ ]:
if __name__ == "__main__":
    import matplotlib.pyplot as plt

    OUT.mkdir(exist_ok=True)
    m1, v1, m2, v2, e = 2.0, 4.0, 1.0, -1.0, 0.5
    a, b = collide(m1, v1, m2, v2, e)
    print(f"v1' = {a:.3f} m/s, v2' = {b:.3f} m/s")
    print(f"momentum before {m1 * v1 + m2 * v2:.2f}, after {m1 * a + m2 * b:.2f} kg m/s")
    print(f"KE before {ke(m1, v1) + ke(m2, v2):.2f} J, after {ke(m1, a) + ke(m2, b):.2f} J, lost {energy_lost(m1, v1, m2, v2, e):.2f} J")
    print(f"ball: e = {restitution_from_heights(1.0, 0.64):.2f};  bullet: {bullet_speed(0.010, 2.0, 0.12):.0f} m/s")

    es = np.linspace(0, 1, 101)
    fig, ax = plt.subplots(figsize=(6.4, 4.2))
    ax.plot(es, [collide(m1, v1, m2, v2, x)[0] for x in es], color="tab:blue", label="body 1 after (2 kg)")
    ax.plot(es, [collide(m1, v1, m2, v2, x)[1] for x in es], color="tab:green", label="body 2 after (1 kg)")
    ax.axhline(0, color="gray", lw=0.8)
    ax.set_xlabel("coefficient of restitution e (no unit)")
    ax.set_ylabel("speed after the collision (m/s)")
    ax.set_title("From sticking (e = 0) to bouncing (e = 1)")
    ax.grid(alpha=0.3)
    ax.legend()
    fig.tight_layout()
    fig.savefig(OUT / "speeds_vs_e.png", dpi=120)

    fig2, ax2 = plt.subplots(figsize=(6.4, 4.2))
    ax2.plot(es, [energy_lost(m1, v1, m2, v2, x) for x in es], color="tab:red")
    ax2.plot(0.5, energy_lost(m1, v1, m2, v2, 0.5), "o", color="black")
    ax2.set_xlabel("coefficient of restitution e (no unit)")
    ax2.set_ylabel("kinetic energy lost (J)")
    ax2.set_title("Energy lost falls to zero as e goes to 1")
    ax2.grid(alpha=0.3)
    fig2.tight_layout()
    fig2.savefig(OUT / "energy_lost_vs_e.png", dpi=120)

## Where this stops
Only the speed along the line of motion is handled. When the bodies hit off-centre the
direction changes too: Session 3 treats collisions of discs in two dimensions.

## Checkpoint
Run this last cell. It prints PASS when your work is correct.

In [ ]:
"""Checkpoint for Session 2. Prints PASS or FAIL."""
import numpy as np


def run_checks(L):
    """L is the lesson module (or notebook namespace). Returns a list of (label, passed)."""
    checks = []
    a, b = L.collide(2, 4, 1, -1, 0.5)
    checks.append((f"worked example: speeds 1.5 and 4.0 m/s (got {a:.3f} and {b:.3f})", abs(a - 1.5) < 1e-9 and abs(b - 4.0) < 1e-9))
    checks.append(("worked example: momentum 7 kg m/s before and after", abs(2 * a + 1 * b - 7) < 1e-9))
    lost = L.energy_lost(2, 4, 1, -1, 0.5)
    checks.append((f"worked example: 6.25 J lost (got {lost:.2f})", abs(lost - 6.25) < 1e-9))
    checks.append(("worked example: lost = KE before - KE after = 16.5 - 10.25 J", abs((L.ke(2, 4) + L.ke(1, -1)) - (L.ke(2, a) + L.ke(1, b)) - lost) < 1e-9))
    checks.append(("e = 0 reproduces the sticking speed", all(abs(x - (2 * 4 + 1 * -1) / 3) < 1e-12 for x in L.collide(2, 4, 1, -1, 0))))
    checks.append(("e = 1 keeps the kinetic energy", abs(L.energy_lost(2, 4, 1, -1, 1.0)) < 1e-12))
    ok = True
    for m1, v1, m2, v2, e in ((2, 4, 1, -1, 0.3), (1, 5, 4, -2, 0.9), (5, 1, 2, 3, 0.0)):
        x, y = L.collide(m1, v1, m2, v2, e)
        ok &= abs((y - x) - e * (v1 - v2)) < 1e-9 and abs(m1 * x + m2 * y - (m1 * v1 + m2 * v2)) < 1e-9
    checks.append(("the speed of separation equals e times the speed of approach, and momentum is kept", ok))
    checks.append(("a 1 m drop and 0.64 m rebound give e = 0.80", abs(L.restitution_from_heights(1.0, 0.64) - 0.8) < 1e-12))
    checks.append((f"a 10 g bullet in a 2 kg block that rises 12 cm has speed 308 m/s (got {L.bullet_speed(0.010, 2.0, 0.12):.0f})", abs(L.bullet_speed(0.010, 2.0, 0.12) - 308.4) < 0.5))
    a2, b2 = L.collide(0.5, 6, 1.5, 0, 0.8)
    checks.append((f"question B: speeds -2.1 and 2.7 m/s, 2.43 J lost (got {a2:.2f}, {b2:.2f}, {L.energy_lost(0.5, 6, 1.5, 0, 0.8):.2f})",
                   abs(a2 + 2.1) < 1e-9 and abs(b2 - 2.7) < 1e-9 and abs(L.energy_lost(0.5, 6, 1.5, 0, 0.8) - 2.43) < 1e-9))
    checks.append(("design: m1 = 4 kg, m2 = 1 kg, v1 = 5 m/s, e = 1 sends body 2 away at 8 m/s with no loss",
                   abs(L.collide(4, 5, 1, 0, 1.0)[1] - 8) < 1e-9 and abs(L.energy_lost(4, 5, 1, 0, 1.0)) < 1e-12))
    for bad, label in ((lambda: L.collide(1, 1, 1, 0, 1.2), "e above 1"), (lambda: L.collide(1, 1, 1, 0, -0.1), "a negative e"),
                       (lambda: L.restitution_from_heights(1.0, 1.5), "a rebound higher than the drop"), (lambda: L.collide(0, 1, 1, 0, 0.5), "mass 0")):
        try:
            bad()
            raised = False
        except ValueError:
            raised = True
        checks.append((f"edge case: {label} gives an error", raised))
    return checks
import types
L = types.SimpleNamespace(**{k: v for k, v in globals().items() if not k.startswith('_')})
ok = True
for label, passed in run_checks(L):
    print(('PASS  ' if passed else 'FAIL  ') + label)
    ok &= bool(passed)
print('PASS' if ok else 'FAIL')